# 02 - Error analysis

Where the system fails, taken from `results/`. Run `make eval` first.

The README leads with rates; this notebook is the 'what do the failures
actually look like' half, which is the part that carries the result.

In [ ]:
import json, pathlib, collections, sys
sys.path.insert(0, '..')
RESULTS = pathlib.Path('../results')
path = RESULTS / 'metrics.json'
if not path.exists():
    path = RESULTS / 'metrics_echo.json'
    print('No model run found; showing the echo baseline instead.')
metrics = json.loads(path.read_text())
print('backend:', metrics['run']['backend'], '| retriever:', metrics['run']['retriever'])

## Retrieval: does the embedding step earn its place?

In [ ]:
for name, arm in metrics['arm1_retrieval'].items():
    r1, r5 = arm['recall@1'], arm['recall@5']
    print(f"{name:14} recall@1 {r1['point']:.3f}  recall@5 {r5['point']:.3f}")
print()
print('If structured >= hybrid, the retrieval layer is a dictionary lookup')
print('wearing a vector-store costume, and the README should say so.')

## The contradictions

Cases where the system asserted the opposite of the held-out label.

In [ ]:
records = metrics['arm3b_leave_one_out_agreement']['records']
wrong = [r for r in records if r['verdict'] == 'contradict']
print(f'{len(wrong)} contradictions out of {len(records)}\n')
for r in wrong[:10]:
    print(f"{r['gene']:8} {str(r['protein_change']):8} "
          f"label={r['label']:22} predicted={r['predicted']:22}")
    if r['reasoning']:
        print(f"    {r['reasoning'][:150]}")

## Which genes fail most

Concentrated failure is a fixable bug; spread-out failure is a ceiling.

In [ ]:
by_gene = collections.Counter(r['gene'] for r in records if r['verdict'] == 'contradict')
for gene, count in by_gene.most_common(12):
    total = sum(1 for r in records if r['gene'] == gene)
    print(f'{gene:10} {count:3} contradictions of {total:3} cases')

## Confabulation on negative controls

The arm nearly every project skips. `hard` = the gene IS in CIViC at other
positions, which is where a plausible-sounding wrong answer comes from.

In [ ]:
arm4 = metrics.get('arm4_negative_controls', {})
for stratum in ('hard', 'easy', 'pooled'):
    data = arm4.get(stratum)
    if not data or not data.get('confabulation_rate', {}).get('denominator'):
        continue
    rate = data['confabulation_rate']
    print(f"{stratum:7} n={data['n']:4}  confabulation "
          f"{rate['point']:.3f} [{rate['low']:.3f}, {rate['high']:.3f}]")
print()
for example in (arm4.get('hard', {}).get('examples') or [])[:8]:
    print(f"  {example['gene']:8} {example['query']:24} "
          f"-> {example['predicted']} {example['citations']}")

## Did retrieval help?

Ablation: the same arms with the context removed.

In [ ]:
with_r = metrics['arm3b_leave_one_out_agreement']
without = metrics['ablation']['arm3b_no_retrieval']
for label, arm in (('with retrieval', with_r), ('no retrieval', without)):
    print(f"{label:15} agreement {arm['agreement_rate']['point']:.3f}  "
          f"contradiction {arm['contradiction_rate']['point']:.3f}  "
          f"net benefit {arm['net_benefit_per_100']}")